# Part 8: Save, load and share

A fine-tuned model is a folder: the weights, a configuration file and the tokenizer files. Anyone can load it from that folder, or from the Hub if you upload it. This part trains a small model (200 examples, three epochs, about a minute on a laptop CPU). The point is the files, but 200 examples is enough for the predictions to mean something. With only a few dozen examples the two scores stay close to 0.5, which means the new layer has learned almost nothing.

In [1]:
import warnings

from transformers.utils import logging

logging.set_verbosity_error()  # hide the long model-loading reports
warnings.filterwarnings("ignore")

from review_data import as_lists, load_reviews, small_sample

reviews = load_reviews()
val_texts, val_labels = as_lists(reviews["validation"])

texts, labels = small_sample(reviews["train"], 200, seed=42)

c:\Users\student\AppData\Local\anaconda3\envs\aigc-5005-module09-hf\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from finetune import fine_tune
from share import save_model

model, tokenizer, _ = fine_tune(texts, labels, val_texts[:64], val_labels[:64], epochs=3)
folder = save_model(model, tokenizer, "saved_model")
print(sorted(p.name for p in folder.iterdir()))

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.28it/s]

['config.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json']


In [3]:
from share import label_scores, load_classifier

classifier = load_classifier("saved_model")
for sentence in ["a warm, funny and beautifully acted film", "dull, predictable and far too long"]:
    print(sentence, label_scores(classifier, sentence))

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 1187.59it/s]


a warm, funny and beautifully acted film {'positive': 0.8568418622016907, 'negative': 0.14315813779830933}
dull, predictable and far too long {'negative': 0.7317764163017273, 'positive': 0.26822352409362793}


## Run the Gradio app

From a terminal in this folder, run `python app.py saved_model`. Gradio prints a local address. Open it, type a sentence, and the app returns the two scores. The same `app.py` runs on a Space once the model is on the Hub.

## Share on the Hub

These two lines upload the model. They need a Hugging Face account and a token with write access, which you create in your account settings and store with `hf auth login`. Never paste a token into a notebook or commit it to Git.

```python
from huggingface_hub import login

login(token="hf_your_write_token_here")

model.push_to_hub("your-username/review-sentiment")
tokenizer.push_to_hub("your-username/review-sentiment")
```

Then write a **model card**: what the model is for, the data it was trained on, how it was evaluated, and where it fails. The Hub shows the card on the model page.



## Try it

1. List the files in `saved_model` and look at `config.json`. Where are the label names?
2. Load the saved folder with `AutoModelForSequenceClassification` instead of a pipeline.
3. Write three sentences for a model card for this model, including one honest limitation.